# 01 - Analyse exploratoire (section 2.1)

Dans ce notebook l'objectif est de décrire les données, vérifier leur qualité, analyser la distribution et le déséquilibre des classes, et observer les relations entre les differentes variables.

**Cible binaire :** `bon = 1` si `quality >= 7`.

In [19]:
import sys
from pathlib import Path

# Fonctionne qu'on lance le notebook depuis notebooks/ ou depuis la racine du projet
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

from src.load_data import DATA_DIR, RED_FILE, WHITE_FILE, load_wine_data, make_binary_target

print("Dossier des données :", DATA_DIR.resolve())

ImportError: cannot import name 'RED_FILE' from 'src.load_data' (C:\Users\abrdr\OneDrive\Documents\ING2\Machine Learning\WineQuality\src\load_data.py)

## 1. Chargement et vérification des sources

In [20]:
for name in (RED_FILE, WHITE_FILE):
    print(f"{name:25s} présent : {(DATA_DIR / name).exists()}")

df = make_binary_target(load_wine_data())
print("\nDimensions après fusion :", df.shape)
df.head()

NameError: name 'RED_FILE' is not defined

## 2. Statistiques descriptives

In [ ]:
df.info()

In [ ]:
df.describe().T.round(3)

## 3. Valeurs manquantes et doublons

In [ ]:
print("Valeurs manquantes au total :", int(df.isna().sum().sum()))
print("Doublons exacts             :", int(df.duplicated().sum()))
print()
print("Doublons par type de vin :")
print(df.assign(dup=df.duplicated()).groupby("type")["dup"].sum())

**À compléter :** nombre de doublons, répartition rouge/blanc, et traitement retenu (suppression dans `clean_data`, au notebook 02).

## 4. Distribution des classes et déséquilibre

In [ ]:
print("Effectifs par note de qualité :")
print(df["quality"].value_counts().sort_index())

fig, ax = plt.subplots(figsize=(8, 4))
sns.countplot(x="quality", hue="type", data=df, ax=ax)
ax.set_title("Distribution des notes de qualité")
plt.tight_layout()
plt.show()

In [ ]:
counts = df["bon"].value_counts().sort_index()
n_moins_bon, n_bon = int(counts.loc[0]), int(counts.loc[1])

print("Effectifs (0 = moins bon, 1 = bon) :", {0: n_moins_bon, 1: n_bon})
print("Proportion de bons vins            :", round(n_bon / len(df), 3))
print("Ratio moins bon / bon             :", round(n_moins_bon / n_bon, 2))
print()
print("Proportion de bons vins par type :")
print(df.groupby("type")["bon"].mean().round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.countplot(x="bon", data=df, ax=axes[0], hue="bon", legend=False, palette="Set2")
axes[0].set_title("Classe binaire (global)")
sns.countplot(x="type", hue="bon", data=df, ax=axes[1], palette="Set2")
axes[1].set_title("Classe binaire par type de vin")
plt.tight_layout()
plt.show()

**À compléter :** commentez le ratio de déséquilibre. Quelles conséquences pour le choix des métriques (accuracy seule trompeuse, intérêt du rappel, du F1 et de l'AUC) ? Les vins blancs sont-ils plus souvent « bons » que les rouges ?

## 5. Histogrammes des variables

In [ ]:
features = [c for c in df.columns if c not in ["type", "quality", "bon"]]

fig, axes = plt.subplots(4, 3, figsize=(15, 16))
for ax, col in zip(axes.flat, features):
    sns.histplot(df[col], kde=True, ax=ax, color="steelblue")
    ax.set_title(col)
for ax in list(axes.flat)[len(features):]:
    ax.set_visible(False)
plt.tight_layout()
plt.show()

**À compléter :** variables asymétriques (queue à droite), variables proches d'une loi normale.

## 6. Valeurs aberrantes (méthode IQR)

Les valeurs aberrantes sont signalées, pas supprimées. Le pipeline les plafonnera (`IQRClipper`), ajusté sur le train uniquement.

In [ ]:
def iqr_bounds(s, k=1.5):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

rows = []
for col in features:
    lo, hi = iqr_bounds(df[col])
    n_out = int(((df[col] < lo) | (df[col] > hi)).sum())
    rows.append({
        "variable": col,
        "borne_basse": lo,
        "borne_haute": hi,
        "n_aberrantes": n_out,
        "pourcentage": 100 * n_out / len(df),
    })

outliers = pd.DataFrame(rows).sort_values("pourcentage", ascending=False)
outliers.round(2)

## 7. Boxplots par classe

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(15, 16))
for ax, col in zip(axes.flat, features):
    sns.boxplot(x="bon", y=col, data=df, ax=ax, hue="bon", legend=False, palette="Set2")
    ax.set_title(col)
    ax.set_xlabel("bon (0 = non, 1 = oui)")
for ax in list(axes.flat)[len(features):]:
    ax.set_visible(False)
plt.tight_layout()
plt.show()

**À compléter :** variables qui séparent le plus les deux classes.

## 8. Matrice de corrélation

In [ ]:
corr = df[features + ["quality"]].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, linewidths=0.5)
plt.title("Matrice de corrélation")
plt.tight_layout()
plt.show()

In [ ]:
print("Corrélation avec la qualité (triées par valeur absolue) :")
corr["quality"].drop("quality").sort_values(key=abs, ascending=False).round(3)

In [ ]:
# Paires de variables fortement corrélées entre elles (|r| > 0.6)
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
pairs = upper.stack().reset_index()
pairs.columns = ["var_1", "var_2", "r"]
pairs[pairs["r"].abs() > 0.6].sort_values("r", key=abs, ascending=False)

**À compléter :** paires redondantes (par exemple `free sulfur dioxide` / `total sulfur dioxide`, ou `density` avec `alcohol` ou `residual sugar`). Pourquoi cela pose-t-il problème pour certains modèles ?

## 9. Synthèse de la section 2.1

- Déséquilibre des classes (ratio et conséquences) :
- Valeurs manquantes et doublons :
- Valeurs aberrantes (à plafonner) :
- Variables les plus liées à la cible :
- Paires de variables redondantes :
- Transformations envisagées (log, etc.) :